In [1]:
from sqlalchemy import Engine, MetaData, Table, cast, func, update
from sqlalchemy.sql.sqltypes import BigInteger, DOUBLE_PRECISION, Integer, SmallInteger, Numeric, Double

import os
from pathlib import Path
import pandas as pd


from sql_utils import drop_table, build_postgres_engine


project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

engine = build_postgres_engine(
        "localhost",
        int(os.environ.get("POSTGRES_PORT", 5432)),
        os.environ["POSTGRES_DB"],
        os.environ["POSTGRES_USER"],
        os.environ["POSTGRES_PASSWORD"],
    )
    
chunck_size = 100_000 # Se estiver estourando mt ram, abaixe esse numero

In [2]:
from sqlalchemy import Column, MetaData, Table, insert, inspect, null, select, String, delete, text
from sqlalchemy.schema import CreateSchema
from sqlalchemy.sql.type_api import TypeEngine

from sql_utils import normalize_table_columns_by_factor

source_schema = "raw"
target_schema_name = "sinan_viol"

def copy_table_between_schemas(
    engine: Engine,
    source_schema_name: str,
    source_table_name: str,
    target_schema_name: str,
    target_table_name: str | None = None,
    column_name_mapping: dict[str, str] | None = None,
    extra_columns: dict[str, TypeEngine] | None = None,
) -> None:
    target_table_name = target_table_name or source_table_name
    column_name_mapping = column_name_mapping or {}
    extra_columns = extra_columns or {}

    inspector = inspect(engine)
    if inspector.has_table(target_table_name, schema=target_schema_name):
        raise ValueError(f"Target table already exists: {target_schema_name}.{target_table_name}")

    source_metadata = MetaData(schema=source_schema_name)
    source_table = Table(source_table_name, source_metadata, autoload_with=engine)

    target_metadata = MetaData(schema=target_schema_name)
    target_columns = [
        Column(
            column_name_mapping.get(column.name, column.name),
            column.type,
            primary_key=column.primary_key,
            nullable=column.nullable,
        )
        for column in source_table.columns
    ]
    target_columns.extend(
        Column(column_name, column_type, nullable=True)
        for column_name, column_type in extra_columns.items()
    )
    target_table = Table(target_table_name, target_metadata, *target_columns)

    source_columns = list(source_table.columns)
    selected_columns = [*source_columns, *[null() for _ in extra_columns]]
    target_column_names = [column.name for column in target_table.columns]

    copy_statement = insert(target_table).from_select(
        target_column_names,
        select(*selected_columns),
    )

    with engine.begin() as connection:
        connection.execute(CreateSchema(target_schema_name, if_not_exists=True))
        target_table.create(connection)
        connection.execute(copy_statement)


def delete_missing_municipality_rows(
    engine: Engine,
    schema_name: str,
    table_name: str,
    territory_code_column: str,
    reference_table_name: str,
    reference_code_column: str,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)
    reference_table = Table(reference_table_name, metadata, autoload_with=engine)

    territory_code = cast(table.c[territory_code_column], String)
    reference_code = cast(reference_table.c[reference_code_column], String)

    statement = delete(table).where(
        func.length(territory_code) > 2,
        territory_code.not_in(select(reference_code)),
    )

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def delete_rows_below_threshold(
    engine: Engine,
    schema_name: str,
    table_name: str,
    column_name: str,
    threshold: int | float,
) -> None:
    metadata = MetaData(schema=schema_name)
    table = Table(table_name, metadata, autoload_with=engine)

    statement = delete(table).where(table.c[column_name] < threshold)

    with engine.begin() as connection:
        result = connection.execute(statement)

    print(f"Deleted rows: {result.rowcount}")

def rename_table_column(
    engine: Engine,
    schema_name: str,
    table_name: str,
    old_column_name: str,
    new_column_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    qualified_table_name = f"{preparer.quote_schema(schema_name)}.{preparer.quote(table_name)}"
    old_column = preparer.quote(old_column_name)
    new_column = preparer.quote(new_column_name)

    statement = text(
        f"ALTER TABLE {qualified_table_name} RENAME COLUMN {old_column} TO {new_column}"
    )

    with engine.begin() as connection:
        connection.execute(statement)

    print(f"Renamed column: {schema_name}.{table_name}.{old_column_name} -> {new_column_name}")

    


## Tirando um copia

In [ ]:
target_table_name = "ocorrencias"
copy_table_between_schemas(engine, source_schema, "SINAN_Violencia", target_schema_name, target_table_name)

In [5]:
from sql_utils import print_table_head

target_table_name = "ocorrencias"

print_table_head(engine, target_schema_name, target_table_name, rows=1000)

        id     TP_NOT ID_AGRAVO DT_NOTIFIC SEM_NOT  NU_ANO SG_UF_NOT  ID_MUNICIP  ID_UNIDADE    DT_OCOR SEM_PRI  ANO_NASC  NU_IDADE_N CS_SEXO                 CS_GESTANT  CS_RACA    CS_ESCOL_N SG_UF  ID_MN_RESI ID_PAIS NDUPLIC DT_INVEST ID_OCUPA_N               SIT_CONJUG SG_UF_OCOR  ID_MN_OCOR HORA_OCOR                 LOCAL_OCOR                     LOCAL_ESPE OUT_VEZES LES_AUTOP VIOL_FISIC VIOL_PSICO VIOL_TORT VIOL_SEXU VIOL_TRAF VIOL_FINAN VIOL_NEGLI VIOL_INFAN VIOL_LEGAL VIOL_OUTR         VIOL_ESPEC AG_FORCA AG_ENFOR AG_OBJETO AG_CORTE AG_QUENTE AG_ENVEN  AG_FOGO AG_AMEACA AG_OUTROS                      AG_ESPEC    SEX_ASSEDI    SEX_ESTUPR SEX_PUDOR     SEX_PORNO     SEX_EXPLO     SEX_OUTRO                   SEX_ESPEC PEN_ORAL PEN_ANAL PEN_VAGINA LESAO_NAT LESAO_ESPE LESAO_CORP   NUM_ENVOLV REL_SEXUAL  REL_PAI  REL_MAE  REL_PAD REL_CONJ REL_EXCON REL_NAMO REL_EXNAM REL_FILHO REL_DESCO REL_IRMAO REL_CONHEC REL_CUIDA REL_PATRAO REL_INST  REL_POL REL_PROPRI REL_OUTROS             REL_E

In [6]:
target_table_name = "ocorrencias"

with engine.connect() as connection:
    inspector = inspect(connection)
    inspector.clear_cache()

    print(f"schema: {target_schema_name}")
    print(f"table: {target_table_name}")

    for column in inspector.get_columns(target_table_name, schema=target_schema_name):
        nullable = "nullable" if column["nullable"] else "not null"
        print(f"  - {column['name']}: {column['type']} ({nullable})")

schema: sinan_viol
table: ocorrencias
  - id: VARCHAR (not null)
  - TP_NOT: TEXT (nullable)
  - ID_AGRAVO: TEXT (nullable)
  - DT_NOTIFIC: TIMESTAMP (nullable)
  - SEM_NOT: TEXT (nullable)
  - NU_ANO: BIGINT (nullable)
  - SG_UF_NOT: TEXT (nullable)
  - ID_MUNICIP: BIGINT (nullable)
  - ID_UNIDADE: BIGINT (nullable)
  - DT_OCOR: TIMESTAMP (nullable)
  - SEM_PRI: TEXT (nullable)
  - ANO_NASC: BIGINT (nullable)
  - NU_IDADE_N: BIGINT (nullable)
  - CS_SEXO: TEXT (nullable)
  - CS_GESTANT: TEXT (nullable)
  - CS_RACA: TEXT (nullable)
  - CS_ESCOL_N: TEXT (nullable)
  - SG_UF: TEXT (nullable)
  - ID_MN_RESI: BIGINT (nullable)
  - ID_PAIS: TEXT (nullable)
  - NDUPLIC: TEXT (nullable)
  - DT_INVEST: TIMESTAMP (nullable)
  - ID_OCUPA_N: TEXT (nullable)
  - SIT_CONJUG: TEXT (nullable)
  - SG_UF_OCOR: TEXT (nullable)
  - ID_MN_OCOR: BIGINT (nullable)
  - HORA_OCOR: TEXT (nullable)
  - LOCAL_OCOR: TEXT (nullable)
  - LOCAL_ESPE: TEXT (nullable)
  - OUT_VEZES: TEXT (nullable)
  - LES_AUTOP: TEXT

In [ ]:
from sqlalchemy import text


def drop_schema(
    engine: Engine,
    schema_name: str,
) -> None:
    preparer = engine.dialect.identifier_preparer
    quoted_schema = preparer.quote_schema(schema_name)

    with engine.begin() as connection:
        connection.execute(text(f"DROP SCHEMA IF EXISTS {quoted_schema} CASCADE"))

    inspector = inspect(engine)
    inspector.clear_cache()
    if inspector.has_schema(schema_name):
        raise RuntimeError(f"Schema still exists after DROP SCHEMA CASCADE: {schema_name}")

    print(f"Dropped schema and all objects inside it: {schema_name}")


# Para usar quando quiser recomeçar:
# drop_schema(engine, target_schema_name)


Dropped schema and all objects inside it: sinan_viol
